# p7: toward a POMDP teacher — a learning model the teacher can plan through

bayes3 is a Bayesian student model with a hand-set threshold rule, not a POMDP solution. The teacher's real problem is a POMDP:

- **hidden state:** what the student knows, k;
- **actions:** show the optimal move, or let the student play;
- **observations:** the student's own moves;
- **transition:** the student's play within a puzzle, and *learning* between updates — the network's update $U_\alpha$ at agency $\alpha$;
- **reward:** the gain on held-out test puzzles at the end of the session. Shaped if needed: the change in expected test score each step, $\Phi(b_{t+1}) - \Phi(b_t)$, which sums to the same thing.

The network's update can't be planned through: one session takes about 25 minutes. So the plan is to fit a small learning model to the sessions already run, check that it reproduces them, and only then solve the POMDP it defines:

1. **Fit a learning model** (this notebook): per-bin test score as the knowledge state, moved by what each stretch of the session gave the student to learn from.
2. **Validate it** (this notebook): fit on some teachers or agencies, predict the others. In particular, it has to reproduce the dilemma from p6: where more help starts to cost at each agency, and random 100% against random 30%.
3. **Refit with logged losses** (this notebook): does an example teach in proportion to its loss?
4. **Diagnose the cost of lost own play** (this notebook): why does the model under-price heavy help at low agency?
5. **Does a θ that changes beat a constant one?** (this notebook, last section): θ switched by time or puzzle length, run on the real student, and the learning model's prediction of it.
6. **An upper bound** (this notebook, last section): a play model (the own and shown examples a teacher's choice produces from a given state), then the best plan if the student's state were known. No POMDP teacher can beat it in this model.
7. Next, only if the bound leaves real room above θ 0.45: solve the POMDP and test its teacher on the real students.

Data: every equal-batch run in p6 (`teach_shortcut_agency{a}_even`, agency 0 to 1) — no teacher, random 30% and 100%, bayes3 θ 0.25 to 0.97. The students are tested every 18 human trials (540 puzzles at multiplier 30), so each session gives 4 intervals.

In [ ]:
import sys
sys.path.append("../scripts")
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

import session_analysis as sa
import teach_experiment as te

FIGURES = "../visualization/p7_pomdp"
os.makedirs(FIGURES, exist_ok=True)
AGENCIES = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
SPECS = ["none", "random:0.3", "random:1.0"] + [f"bayes3:{t}" for t in (0.25, 0.45, 0.75, 0.9, 0.97)]
BINS = [(9, 11), (14, 16), (19, 21), (24, 26)]  # the session's (and tests') length bins
NB = len(BINS)
AGENCY_COLORS = {0.0: "#93bfdc", 0.1: "#73a9d3", 0.3: "#5393c7", 0.5: "#3a7cb9", 0.7: "#2565a3", 0.9: "#154f88",
                 1.0: "#08306b"}  # as in p6: one hue, light (low agency) to dark


def run_of(a):
    "The p6 equal-batch run at agency a (no teacher is saved in agency 0's only: it shows nothing)."
    return "teach_shortcut_agency1_even" if a == 1.0 else f"teach_shortcut_agency{a}_even"


def bin_of(d):
    return next(i for i, (lo, hi) in enumerate(BINS) if lo <= d <= hi)

## The data: what each interval gave the student to learn from

How the student learns (`rl.learning_session`, `rl.intuition_examples`, `rl.agency_weights`, `rl.online_update`):

- A **solved** puzzle teaches its boards, labelled with shortest routes through the boards visited (labels "all"). An **unsolved** one teaches only the moves the teacher showed.
- Examples on boards where a move was shown weigh α in the loss; the rest weigh 1.
- Every 30 puzzles the pooled examples get 3 passes in batches of up to 256. With agency step "skip", each batch's step is taken with probability equal to its mean weight.

So, for one puzzle, expected gradient work is proportional to its examples times their weight. Per interval and length bin I count:

- **own examples:** a solved puzzle's examples times the share of its moves the student played itself;
- **shown examples:** the rest of a solved puzzle's examples, and all of an unsolved puzzle's.

The share of a solved puzzle's examples that sit on shown boards is taken as its share of shown moves. That's an approximation: `agency_weights` also gives weight α to some destinations on earlier boards.

Knowledge state: the test score per bin (moves / optimal, an unsolved attempt counted at its cap; lower is better), 50 puzzles × 3 attempts per bin.

In [ ]:
def bin_scores(attempts, config):
    "Test score per length bin."
    total, n = np.zeros(NB), np.zeros(NB)
    for x in attempts:
        b = bin_of(x["distance"])
        total[b] += (x["moves"] if x["solved"] else te.step_cap(x["distance"], config)) / x["distance"]
        n[b] += 1
    return total / n


def intervals():
    "One row per (agency, teacher, student, interval between tests): per-bin scores at both ends and the material."
    rows = []
    for a in AGENCIES:
        for spec in SPECS:
            e = sa.entry(run_of(a), spec)
            if e is None:
                continue
            for s, record in e["sessions"].items():
                tests = e["results"][s]
                points = sorted(tests)
                scores = {n: bin_scores(tests[n], e["config"]) for n in points}
                for n0, n1 in zip(points, points[1:]):
                    row = {"agency": a, "teacher": spec, "student": s, "start": n0}
                    chunk = record[n0:n1]
                    for b in range(NB):
                        row[f"s0_{b}"], row[f"s1_{b}"] = scores[n0][b], scores[n1][b]
                        inb = [x for x in chunk if bin_of(x["distance"]) == b]
                        row[f"own_{b}"] = sum(x["examples"] * (1 - x["taught"] / x["moves"]) for x in inb if x["solved"])
                        row[f"mixed_{b}"] = sum(x["examples"] * (1 - x["taught"] / x["moves"]) for x in inb
                                                if x["solved"] and x["taught"])
                        row[f"shown_{b}"] = sum(x["examples"] * (x["taught"] / x["moves"] if x["solved"] else 1)
                                                for x in inb)
                    row["shown share"] = sum(x["taught"] for x in chunk) / sum(x["moves"] for x in chunk)
                    rows.append(row)
    df = pd.DataFrame(rows)
    df["family"] = df.teacher.str.split(":").str[0]
    return df


data = intervals()
print(len(data), "intervals:", data.groupby(["agency", "teacher"]).ngroups, "conditions x 8 students x 4")
cols = lambda k: [f"{k}_{b}" for b in range(NB)]
summary = data.assign(own=data[cols("own")].sum(axis=1), shown=data[cols("shown")].sum(axis=1),
                      gain=data[cols("s0")].mean(axis=1) - data[cols("s1")].mean(axis=1))
display(summary[summary.agency.isin([0.0, 0.1, 1.0])].groupby(["agency", "teacher"])[["own", "shown", "gain"]]
        .mean().round(2).rename(columns={"own": "own examples / interval", "shown": "shown examples / interval",
                                          "gain": "score gain / interval"}))

## The learning model

Per length bin b, from one test to the next:

$$M_b = \text{own}_b + c\,\alpha^{\gamma}\,\text{shown}_b, \qquad X_b = \sum_{b'} \rho^{|b-b'|} M_{b'} / 1000,$$
$$s_b' = s_b^\infty + (s_b - s_b^\infty)\, e^{-k_b X_b^{\,q}}.$$

- $M_b$ is the learning material: own examples count 1, shown ones $c\,\alpha^\gamma$. With $c = \gamma = 1$ this is exactly the loss weighting; $c$ lets a shown example be worth more or less than an own one (shown moves follow a shortest path), and $\gamma$ how fast that falls with agency.
- $\rho$ is transfer between neighbouring length bins.
- The score approaches a floor $s^\infty_b$ at rate $k_b$; $q < 1$ makes returns diminish within an interval.

12 parameters, fit by least squares to the per-bin scores at the end of each interval (1440 intervals × 4 bins). For comparison:

- **mixed:** one more parameter, δ. Own examples on a solved puzzle that also had shown moves count $1 - \delta(1-\alpha)$. This stands in for what `agency_weights` does and the counts above miss: a shown move also lowers the weight of some destinations on the student's own boards on that path.
- **plain:** the loss weighting as it is ($c = \gamma = 1$).
- **blind:** blind to help; every interval has the same material.

In [ ]:
P_NAMES = [f"floor {lo}-{hi}" for lo, hi in BINS] + [f"rate {lo}-{hi}" for lo, hi in BINS] + ["q", "rho", "c", "gamma",
                                                                                            "delta"]
DIST = np.abs(np.subtract.outer(range(NB), range(NB)))


def unpack(p):
    "(floors, rates, q, rho, c, gamma, delta) from the raw parameter vector."
    return (p[:NB], np.exp(p[NB:2 * NB]), p[2 * NB], 1 / (1 + np.exp(-p[2 * NB + 1])), np.exp(p[2 * NB + 2]),
            p[2 * NB + 3], p[2 * NB + 4])


MATERIALS = {}  # more kinds of material, added further down: {name: function(p, d)}


def material(p, d, kind="full"):
    "Learning material per bin, (rows, bins)."
    *_, c, gamma, delta = unpack(p)
    if kind.split()[0] in MATERIALS:
        return MATERIALS[kind.split()[0]](p, d)
    if kind == "blind":
        return np.full((len(d), NB), 5000.0)
    a = d.agency.values[:, None]
    own = d[cols("own")].values - delta * (1 - a) * d[cols("mixed")].values
    return own + c * a ** gamma * d[cols("shown")].values


def step(p, s0, m):
    "Per-bin scores after an interval with material m."
    floor, rate, q, rho, *_ = unpack(p)  # (c, gamma, delta enter through the material)
    x = (m / 1000) @ rho ** DIST
    return floor + (s0 - floor) * np.exp(-rate * x ** q)


def start(kind="full"):
    # kind: full (delta = 0), mixed (delta free), plain (c = gamma = 1, delta = 0) or blind; a kind ending in
    # " plain" has c = gamma = 1 too; delta is free only in mixed and split (MATERIALS, further down)
    "Starting point and bounds of the raw parameters for a kind of material."
    x0 = np.r_[np.full(NB, 3.0), np.full(NB, -2.0), 0.7, 0.0, 0.0, 1.0, 0.3]
    lo = np.r_[np.zeros(NB), np.full(NB, -12.0), 0.1, -8, -3, 0.2, 0.0]
    hi = np.r_[np.full(NB, 9.0), np.full(NB, 5.0), 3, 8, 3, 5, 1.0]
    if kind != "mixed" and not kind.startswith("split"):  # delta = 0
        lo[-1:], hi[-1:], x0[-1:] = [0.0], [1e-9], [0.0]
    if kind in ("plain", "blind") or kind.endswith(" plain"):  # c = 1, gamma = 1, delta = 0
        lo[-3:], hi[-3:], x0[-3:] = [-1e-9, 1 - 1e-9, 0.0], [1e-9, 1 + 1e-9, 1e-9], [0.0, 1.0, 0.0]
    return x0, lo, hi


def fit(d, kind="full"):
    "Least-squares parameters on the intervals d, each from its observed starting scores."
    x0, lo, hi = start(kind)
    s0, s1 = d[cols("s0")].values, d[cols("s1")].values
    return least_squares(lambda p: (step(p, s0, material(p, d, kind)) - s1).ravel(), x0, bounds=(lo, hi)).x


def sessions(p, d, kind="full"):
    # predicted: chain the model from the first test through each interval's observed material
    "Per session: learning over the session (mean over bins), predicted and observed."
    out = {}
    for key, g in d.groupby(["agency", "teacher", "student"]):
        g = g.sort_values("start")
        s, m = g[cols("s0")].values[:1], material(p, g, kind)
        for i in range(len(g)):
            s = step(p, s, m[i:i + 1])
        first, last = g[cols("s0")].values[0], g[cols("s1")].values[-1]
        out[key] = {"predicted": (first - s[0]).mean(), "observed": (first - last).mean()}
    out = pd.DataFrame(out).T
    out.index.names = ["agency", "teacher", "student"]
    return out


def conditions(per_session):
    "Mean per (agency, teacher)."
    return per_session.astype(float).groupby(["agency", "teacher"]).mean()


def scorecard(name, per_session):
    "Error and correlation of predicted against observed learning per condition; 'within agency': agency means removed."
    m = conditions(per_session)
    err = m.predicted - m.observed
    w = m.groupby("agency").transform(lambda v: v - v.mean())
    return {"test": name, "bias": err.mean(), "rmse": np.sqrt((err ** 2).mean()),
            "r": np.corrcoef(m.predicted, m.observed)[0, 1], "r within agency": np.corrcoef(w.predicted, w.observed)[0, 1]}


def r2(p, d, kind="full"):
    "Share of the variance in per-bin score gains the model explains."
    s0, s1 = d[cols("s0")].values, d[cols("s1")].values
    res = step(p, s0, material(p, d, kind)) - s1
    g = s1 - s0
    return 1 - (res ** 2).sum() / ((g - g.mean(0)) ** 2).sum()


fits = {kind: fit(data, kind) for kind in ("full", "mixed", "plain", "blind")}
floor, rate, q, rho, c, gamma, _ = unpack(fits["full"])
display(pd.Series(np.r_[floor, rate, q, rho, c, gamma], index=P_NAMES[:-1], name="full model").round(3).to_frame().T)
print(f"mixed: delta {unpack(fits['mixed'])[-1]:.2f}")
display(pd.DataFrame([{"model": kind, "R² of interval gains": r2(p, data, kind),
                       **{k: v for k, v in scorecard("in sample", sessions(p, data, kind)).items() if k != "test"}}
                      for kind, p in fits.items()]).set_index("model").round(3))

**Result: the fit.**

- **What drives learning is agency-weighted material, per length bin, with shown examples worth more than own ones.** At agency 1 a shown example is worth $c \approx 2.5$ own ones; a shown move is on a shortest path, so its label is clean. That value falls about in proportion to agency ($\gamma \approx 1.05$). So a shown example is worth an own one at agency about 0.4 (2.47 × 0.4^1.05 ≈ 1), worth less below that and more above.
- **Little transfer between length bins** ($\rho \approx 0.2$): help on long puzzles mostly helps long puzzles.
- **Returns diminish within a bin** through the approach to a floor (1.8, 2.7, 4.4, 4.4 for the four bins), not through $q$, which is close to 1.
- **Against the simpler models,** on learning over the session per condition: r 0.95 against 0.86 for the plain loss weighting and 0 for the help-blind model. The help-blind model still explains 59% of the variance in interval gains. Most of an interval's gain comes from where the student is (early intervals gain more), whatever the help. Help accounts for the rest, and for all of the differences between teachers.
- **δ doesn't earn its place.** The mixed variant fits δ ≈ 0.19 and gains nothing in sample (r 0.956 against 0.953). It also predicts held-out teachers slightly worse (next section). The main model leaves it out.

## Validation: predict teachers and agencies the model wasn't fit on

Each test fits the model without some conditions, then predicts their learning over the whole session: start from the first test and chain the model through the four intervals, using each interval's observed material. So it predicts learning *given the help actually delivered*; predicting the help itself is the play model's job (next step).

- **none + random → bayes3:** fit on no teacher and the random teachers only, predict every bayes3 θ at every agency. The hardest test: bayes3's help is targeted, random help isn't.
- **bayes3 → none + random:** the reverse.
- **leave one agency out**, **leave one teacher out** (all its agencies).

Then the signatures of the dilemma from p6, observed against predicted (leave one teacher out).

In [ ]:
def held_out(train, test, d=data, kind="full", fitter=None):
    "Fit where train(d) holds, predict sessions where test(d) holds; each a boolean mask function. fitter: fit."
    return sessions((fitter or fit)(d[train(d)], kind), d[test(d)], kind)


tests = {
    "none + random → bayes3": held_out(lambda d: d.family != "bayes3", lambda d: d.family == "bayes3"),
    "bayes3 → none + random": held_out(lambda d: d.family == "bayes3", lambda d: d.family != "bayes3"),
    "leave one agency out": pd.concat([held_out(lambda d, a=a: d.agency != a, lambda d, a=a: d.agency == a)
                                       for a in AGENCIES]),
    "leave one teacher out": pd.concat([held_out(lambda d, t=t: d.teacher != t, lambda d, t=t: d.teacher == t)
                                        for t in SPECS]),
    "leave one teacher out, mixed": pd.concat([held_out(lambda d, t=t: d.teacher != t, lambda d, t=t: d.teacher == t,
                                                        kind="mixed") for t in SPECS]),
}
display(pd.DataFrame([scorecard(name, s) for name, s in tests.items()]).set_index("test").round(3))


def signatures(per_session):
    "The dilemma's signatures per agency, from predicted and observed learning per condition."
    m = conditions(per_session)
    out = {}
    for col in ("observed", "predicted"):
        x = m[col].unstack("teacher")
        out[("θ 0.9 − θ 0.45", col)] = x["bayes3:0.9"] - x["bayes3:0.45"]
        out[("θ 0.97 − θ 0.45", col)] = x["bayes3:0.97"] - x["bayes3:0.45"]
        out[("plateau (θ 0.25-0.75)", col)] = x[["bayes3:0.25", "bayes3:0.45", "bayes3:0.75"]].mean(axis=1)
        out[("random 100% − 30%", col)] = x["random:1.0"] - x["random:0.3"]
    return pd.DataFrame(out)


loto = tests["leave one teacher out"]
display(signatures(loto).round(2))

In [ ]:
def validation_figure(per_session, path):
    "Left: predicted against observed learning per condition. Right: the dilemma's signatures by agency."
    m = conditions(per_session)
    sig = signatures(per_session)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
    lim = (min(0.5, m.min().min() - 0.1), 2.35)
    a1.plot(lim, lim, color="#b5b4ad", lw=1, zorder=0)
    markers = {"none": "s", "random": "^", "bayes3": "o"}
    for (a, t), row in m.iterrows():
        fam = t.split(":")[0]
        a1.plot(row.observed, row.predicted, markers[fam], color=AGENCY_COLORS[a], ms=7, mec="white", mew=1)
    for fam, mk in markers.items():
        a1.plot([], [], mk, color="#5f5e5a", label={"none": "no teacher", "random": "random 30%, 100%",
                                                     "bayes3": "bayes3 θ 0.25-0.97"}[fam])
    a1.set_xlim(lim), a1.set_ylim(lim)
    a1.set_xlabel("Observed learning (test score gain)")
    a1.set_ylabel("Predicted, teacher held out")
    a1.legend(frameon=False, fontsize=8.5, loc="upper left")
    a1.set_title("Each condition (shade: agency, light 0 to dark 1)", fontsize=10, loc="left")
    styles = {"θ 0.9 − θ 0.45": "#2a78d6", "θ 0.97 − θ 0.45": "#08306b", "random 100% − 30%": "#1f1f1d"}
    for name, color in styles.items():
        for col, ls in (("observed", "-"), ("predicted", "--")):
            v = sig[(name, col)].dropna()
            a2.plot(v.index, v.values, ls, marker="o" if col == "observed" else None, color=color, lw=2, ms=6,
                    mec="white", mew=1, label=f"{name}" if col == "observed" else None)
    a2.plot([], [], "-", color="#8a8a85", label="observed"), a2.plot([], [], "--", color="#8a8a85", label="predicted")
    a2.axhline(0, color="#5f5e5a", lw=1)
    a2.set_xticks(AGENCIES, [f"{a:g}" for a in AGENCIES])
    a2.set_xlabel("Agency")
    a2.set_ylabel("Learning difference")
    a2.legend(frameon=False, fontsize=8.5, loc="lower right", ncol=2)
    a2.set_title("Where more help costs: observed and predicted", fontsize=10, loc="left")
    for ax in (a1, a2):
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#b5b4ad")
        ax.tick_params(colors="#5f5e5a")
        ax.grid(axis="y", color="#e6e5e0", lw=0.8)
        ax.set_axisbelow(True)
    fig.suptitle("Learning model, each teacher held out in turn: 8 students, mixed lengths, multiplier 30, "
                 "equal batches", fontsize=10, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


validation_figure(loto, os.path.join(FIGURES, "learning_model_validation.png"))

**Result: the model gets every signature of the dilemma in the right direction, with the size roughly right but sometimes off by up to 2×. That's enough to go on to a play model, but not to trust its answer about how much help is best at low agency.**

- **It predicts learning for teachers and agencies it never saw.** Per condition: r 0.94 and error 0.10 when each teacher, or each agency, is held out. Fit on no teacher and the random teachers only, it predicts bayes3 at r 0.91, and orders bayes3's θ within an agency at r 0.87. It does under-predict bayes3's learning by 0.11 overall: targeted help teaches somewhat more per example than the counts credit.
- **The four signatures, observed against predicted (each teacher held out):**

  | | observed | predicted |
  |---|---|---|
  | θ 0.9 − θ 0.45, agency 0 → 1 | −0.32 → +0.06 | −0.10 → −0.01 |
  | θ 0.97 − θ 0.45, agency 0.5 → 1 | −0.30 → −0.11 | −0.15 → −0.03 |
  | plateau (θ 0.25-0.75), agency 0 → 1 | 2.03 → 2.15 | 1.99 → 2.05 |
  | random 100% − 30%, agency 0.1 → 1 | −0.87 → +0.04 | −1.29 → 0.00 |

  - Every signature has the right sign and fades with agency where observed.
  - The cost of bayes3's heavy help, and the plateau's rise with agency, come out at about half to a third of observed.
  - The random gap is too large at low agency. Random 100% on its own is predicted well (0.50 against 0.63 at agency 0.1). The miss is random 30%, predicted 1.79 against 1.51 observed: help mixed into own play costs more than the counts show.
  - δ was meant to capture that, but it only half-fixes the random gap and doesn't touch bayes3's θ 0.9 cost.
- **What the model is missing:** at low agency, heavy *targeted* help costs more than the material it removes. A likely reason is that what an example teaches depends on the board. bayes3 at high θ shows exactly the boards the student would get wrong, which are the most informative ones. Random help shows boards the student knows as often as ones it doesn't. The counts treat every example alike.

**Caveats.**
- The model predicts learning from the help actually delivered. Predicting how much help a policy delivers, from the student's state, is the play model (step 3).
- The shown share of a solved puzzle's examples is approximated by its share of shown moves. The runs don't save the exact loss weights.
- The model is fit on 8 students and 45 conditions, all at multiplier 30 with mixed lengths and full hindsight.

**Next.**
- Option 1: refine the material before building on it. Weight each example by how likely the student was to get it wrong (bayes3's own solve-alone estimate is saved per puzzle for bayes3 runs, but not for random ones). Or log the exact loss weight per puzzle in `learning_session` for future runs (done: see the refit below).
- Option 2: go on to step 3 with this model. Expect a POMDP solved on it to show *more* than is best at low agency, since it under-prices heavy targeted help.

## Refit: does what an example teaches depend on its loss?

The model above leaves two questions open. Why is a shown example worth $c \approx 2.5$ own ones at agency 1? And why does heavy targeted help (bayes3 θ 0.9) cost more at low agency than the material it removes? One candidate answer to both: an example teaches in proportion to its loss. bayes3 shows exactly the boards the student would get wrong, so its shown examples should carry high loss, and counting examples alike undervalues them.

To test this, `rl.learning_session(log_loss=True)` (`session_experiment --log-loss`) now records, for every puzzle, the loss of each of its examples, measured on the network just before the update that learns from them (eval mode, no gradient, so training is unchanged). The sums are split into **own** examples and examples on **shown** boards. It also records each update's batches and the steps actually taken. Queue `loss_log` re-ran 20 conditions with the logging on: no teacher, random 30% and 100%, bayes3 θ 0.45, 0.9 and 0.97, at agencies 0, 0.1, 0.3, 0.5 and 1. These are fresh samples of 8 students each: earlier runs can't be replayed exactly, because example order depended on Python's per-process hash seed. That seed is now fixed.

Kinds of material, each fit with $c$ and $\gamma$ free and as **plain** ($c = \gamma = 1$, the loss weighting as it stands):

- **count:** examples, as above. Own and shown are split by share of moves shown, as above.
- **exact:** examples, split by the logged own/shown kind.
- **loss:** each example's training loss times its weight, summed: policy + destination (per target, as the batch averages them) + ½ value. Shown examples carry weight α, so plain loss material is exactly the weighted loss the network minimizes.
- **policy:** the policy term alone.
- **steps:** the Adam steps actually taken, shared out over a block's examples by their share of its weighted loss. Adam moves about the same distance each step, whatever the loss, so this asks whether steps rather than loss carry the learning.
- **split:** own examples split into an optimal path's worth (a solved puzzle's optimal length times its share of own moves), counting 1, and the wandering beyond it, counting δ. This tests a second explanation for $c$: own example counts swell with wandering, while shown moves sit on a shortest path.

In [ ]:
LOSS_AGENCIES = [0.0, 0.1, 0.3, 0.5, 1.0]
LOSS_SPECS = ["none", "random:0.3", "random:1.0", "bayes3:0.45", "bayes3:0.9", "bayes3:0.97"]
LOSS_KINDS = ("own", "shown")


def training_loss(sums, per_target):
    "An example kind's loss terms as _weighted_loss adds them: policy + dest per target + ½ value; weighted, unweighted."
    return (sums["w_policy"] + sums["w_dest"] / per_target + 0.5 * sums["w_value"],
            sums["policy"] + sums["dest"] / per_target + 0.5 * sums["value"])


def loss_intervals():
    # per (agency, teacher, student, interval): the scores, the count features as in intervals(), and per bin and
    # kind (own, shown): exact examples n, weighted loss l, unweighted loss u, weighted policy loss p, steps s
    # (a block's steps shared by weighted loss); ownd: a path's worth of own examples
    "Intervals of the loss_log runs, with logged losses and steps."
    found = [(a, spec, e) for a in LOSS_AGENCIES for spec in LOSS_SPECS
             if (e := sa.entry(f"loss_log_agency{a}", spec)) is not None]
    sums = [x["loss"][k] for _, _, e in found for r in e["sessions"].values() for x in r for k in x.get("loss", {})]
    per_target = sum(s["n_dest"] for s in sums) / sum(s["n"] for s in sums)  # dest targets per example
    rows = []
    for a, spec, e in found:
        block = e["config"]["block"]
        for s, record in e["sessions"].items():
            tests = e["results"][s]
            points = sorted(tests)
            scores = {n: bin_scores(tests[n], e["config"]) for n in points}
            for n0, n1 in zip(points, points[1:]):
                row = {"agency": a, "teacher": spec, "student": s, "start": n0}
                f = {k: np.zeros(NB) for k in ("own", "mixed", "shown", "ownd") + tuple(
                    k + x for k in LOSS_KINDS for x in "nlups")}
                chunk = record[n0:n1]
                for x in chunk:
                    b = bin_of(x["distance"])
                    own_share = 1 - x["taught"] / x["moves"]
                    f["own"][b] += x["examples"] * own_share if x["solved"] else 0
                    f["mixed"][b] += x["examples"] * own_share if x["solved"] and x["taught"] else 0
                    f["shown"][b] += x["examples"] * (1 - own_share if x["solved"] else 1)
                    f["ownd"][b] += x["distance"] * own_share if x["solved"] else 0
                for i in range(n0, n1, block):  # one update per block: its steps, shared by weighted loss
                    blk = record[i:i + block]
                    total = sum(training_loss(x["loss"][k], per_target)[0] for x in blk for k in x.get("loss", {}))
                    for x in blk:
                        b = bin_of(x["distance"])
                        for k, sm in x.get("loss", {}).items():
                            lw, lu = training_loss(sm, per_target)
                            f[k + "n"][b] += sm["n"]
                            f[k + "l"][b] += lw
                            f[k + "u"][b] += lu
                            f[k + "p"][b] += sm["w_policy"]
                            f[k + "s"][b] += blk[0]["update"]["steps"] * lw / total if total > 0 else 0
                for b in range(NB):
                    row[f"s0_{b}"], row[f"s1_{b}"] = scores[n0][b], scores[n1][b]
                    row.update({f"{k}_{b}": v[b] for k, v in f.items()})
                row["shown share"] = sum(x["taught"] for x in chunk) / sum(x["moves"] for x in chunk)
                rows.append(row)
    df = pd.DataFrame(rows)
    df["family"] = df.teacher.str.split(":").str[0]
    return df, per_target


logged, PER_TARGET = loss_intervals()
print(len(logged), "intervals:", logged.groupby(["agency", "teacher"]).ngroups, "conditions x 8 students x 4;",
      f"{PER_TARGET:.1f} destination targets per example")
tot = lambda k: logged[cols(k)].sum(axis=1)
per_example = logged.assign(own=tot("ownu") / tot("ownn").replace(0, np.nan),
                            shown=tot("shownu") / tot("shownn").replace(0, np.nan),
                            steps=tot("owns") + tot("showns"), own_ex=tot("ownn"), shown_ex=tot("shownn"))
per_example = per_example.groupby(["agency", "teacher"])[["own_ex", "shown_ex", "own", "shown", "steps"]].mean()
per_example["shown / own"] = per_example["shown"] / per_example["own"]
display(per_example.round(2).rename(columns={
    "own_ex": "own examples", "shown_ex": "shown examples", "own": "loss per own example",
    "shown": "loss per shown example", "steps": "steps taken"}))

In [ ]:
def weighted_material(own, shown, alpha_weighted):
    # own + c alpha^(gamma - 1) * shown, with shown carrying its loss weight alpha (alpha_weighted: multiply counts
    # by alpha here; loss and steps have it already). For counts this is own + c alpha^gamma shown, as above.
    # Scaled so the material averages what the counts do.
    a_all = logged.agency.values[:, None]
    scale = (logged[cols("own")].values + a_all * logged[cols("shown")].values).sum() / (
        logged[cols(own)].values + logged[cols(shown)].values * (a_all if alpha_weighted else 1)).sum()

    def m(p, d):
        *_, c, gamma, _ = unpack(p)
        a = d.agency.values[:, None]
        s = d[cols(shown)].values * (a if alpha_weighted else 1)
        return scale * (d[cols(own)].values + c * np.where(a > 0, np.maximum(a, 1e-12) ** (gamma - 1), 0) * s)
    return m


def split_material(p, d):
    "Own examples: a path's worth counts 1, the wandering beyond it delta; shown examples as counted."
    *_, c, gamma, delta = unpack(p)
    a = d.agency.values[:, None]
    path, own = d[cols("ownd")].values, d[cols("own")].values
    return path + delta * np.maximum(own - path, 0) + c * np.where(a > 0, np.maximum(a, 1e-12) ** gamma, 0) * d[
        cols("shown")].values


MATERIALS.update({"count": weighted_material("own", "shown", True), "exact": weighted_material("ownn", "shownn", True),
                  "loss": weighted_material("ownl", "shownl", False),
                  "policy": weighted_material("ownp", "shownp", False),
                  "steps": weighted_material("owns", "showns", False), "split": split_material})
REFIT_KINDS = ["count", "count plain", "exact", "exact plain", "loss", "loss plain", "policy", "policy plain",
               "steps", "steps plain", "split", "blind"]


def refit_row(kind):
    "Parameters and scores of one kind of material on the logged intervals."
    p = fit(logged, kind)
    *_, q, rho, c, gamma, delta = unpack(p)
    loto = pd.concat([held_out(lambda d, t=t: d.teacher != t, lambda d, t=t: d.teacher == t, logged, kind)
                      for t in LOSS_SPECS])
    to_b3 = held_out(lambda d: d.family != "bayes3", lambda d: d.family == "bayes3", logged, kind)
    inside, out, b3 = (scorecard("", s) for s in (sessions(p, logged, kind), loto, to_b3))
    return loto, {"material": kind, "c": c, "γ": gamma, "δ": delta if kind.startswith("split") else np.nan, "q": q,
                  "ρ": rho, "R² of interval gains": r2(p, logged, kind), "r": inside["r"], "rmse": inside["rmse"],
                  "r, teacher out": out["r"], "rmse, teacher out": out["rmse"], "r → bayes3": b3["r"],
                  "r → bayes3 within agency": b3["r within agency"], "bias → bayes3": b3["bias"]}


refit_loto, refit_rows = {}, []
for kind in REFIT_KINDS:
    refit_loto[kind], row = refit_row(kind)
    refit_rows.append(row)
refit_table = pd.DataFrame(refit_rows).set_index("material")
display(refit_table.round(3))

SIG_NAMES = ("θ 0.9 − θ 0.45", "θ 0.97 − θ 0.45", "random 100% − 30%", "bayes3 θ 0.45")


def logged_signatures(per_session):
    "The dilemma's signatures the logged conditions have, per agency."
    m = conditions(per_session)
    out = {}
    for col in ("observed", "predicted"):
        x = m[col].unstack("teacher")
        for name, v in zip(SIG_NAMES, (x["bayes3:0.9"] - x["bayes3:0.45"], x["bayes3:0.97"] - x["bayes3:0.45"],
                                       x["random:1.0"] - x["random:0.3"], x["bayes3:0.45"])):
            out[(name, col)] = v
    return pd.DataFrame(out)


sig = pd.DataFrame({(n, "observed"): logged_signatures(refit_loto["count"])[(n, "observed")] for n in SIG_NAMES})
for kind in ("count", "loss", "steps"):
    s = logged_signatures(refit_loto[kind])
    for n in SIG_NAMES:
        sig[(n, kind)] = s[(n, "predicted")]
sig = sig[[(n, c) for n in SIG_NAMES for c in ("observed", "count", "loss", "steps")]]
display(sig.round(2))

In [ ]:
def refit_figure(path):
    "Left: loss per example on shown against own boards, per condition. Right: θ 0.9's cost, observed and predicted."
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
    teachers = [t for t in LOSS_SPECS if t != "none"]
    names = {"random:0.3": "random 30%", "random:1.0": "random 100%", "bayes3:0.45": "bayes3 θ 0.45",
             "bayes3:0.9": "bayes3 θ 0.9", "bayes3:0.97": "bayes3 θ 0.97"}
    for (a, t), row in per_example.iterrows():
        if t == "none":
            continue
        x = teachers.index(t) + (LOSS_AGENCIES.index(a) - 2) * 0.09
        if not np.isnan(row["own"]):
            a1.plot(x, row["own"], "o", color=AGENCY_COLORS[a], mfc="white", mew=1.6, ms=7)
        a1.plot(x, row["shown"], "o", color=AGENCY_COLORS[a], mec="white", mew=1, ms=8)
    a1.plot([], [], "o", color="#5f5e5a", mfc="white", mew=1.6, label="own boards")
    a1.plot([], [], "o", color="#5f5e5a", mec="white", label="shown boards")
    a1.set_xticks(range(len(teachers)), [names[t] for t in teachers])
    a1.set_ylabel("Loss per example, before its update")
    a1.legend(frameon=False, fontsize=8.5, loc="upper right")
    a1.set_title("bayes3 shows high-loss boards; random help doesn't (shade: agency)", fontsize=10, loc="left")
    styles = {"observed": ("-", "o", "#08306b"), "count": ("--", None, "#2a78d6"), "loss": (":", None, "#2a78d6"),
              "steps": ("-.", None, "#2a78d6")}
    for kind, (ls, mk, color) in styles.items():
        v = sig[("θ 0.9 − θ 0.45", kind)].dropna()
        a2.plot(v.index, v.values, ls, marker=mk, color=color, lw=2, ms=6, mec="white", mew=1,
                label=kind if kind == "observed" else f"predicted from {kind}")
    a2.axhline(0, color="#5f5e5a", lw=1)
    a2.set_xticks(LOSS_AGENCIES, [f"{a:g}" for a in LOSS_AGENCIES])
    a2.set_xlabel("Agency")
    a2.set_ylabel("Learning, θ 0.9 minus θ 0.45")
    a2.legend(frameon=False, fontsize=8.5, loc="lower right")
    a2.set_title("The cost of heavy targeted help, each teacher held out", fontsize=10, loc="left")
    for ax in (a1, a2):
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#b5b4ad")
        ax.tick_params(colors="#5f5e5a")
        ax.grid(axis="y", color="#e6e5e0", lw=0.8)
        ax.set_axisbelow(True)
    fig.suptitle("Loss-logged runs (queue loss_log): 8 students, mixed lengths, multiplier 30, equal batches",
                 fontsize=10, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


refit_figure(os.path.join(FIGURES, "loss_refit.png"))

**Result: loss doesn't explain why shown examples are worth more, or why heavy targeted help costs. Counting examples is still the best material.**

- **The logged runs reproduce p6 and the model above.** These are fresh samples, and the count model refit on them gives $c$ 2.50 and $\gamma$ 0.84 (above: 2.47 and 1.05). The signatures come out about as in p6:
  - θ 0.9 − θ 0.45: −0.24, −0.17, −0.23, −0.13, +0.01 at agencies 0, 0.1, 0.3, 0.5 and 1. In p6: −0.32, −0.15, −0.19, −0.03, +0.06.
  - random 100% − 30%: −0.90, −0.31, +0.07 at agencies 0.1, 0.5 and 1. In p6: −0.87, −0.25, +0.04.
- **The losses differ as expected.** bayes3's shown boards carry 1.4 to 1.7 times the loss of the student's own boards: 2.0 to 2.4 per example against about 1.4. Random 30%'s shown boards carry only 1.07 times its own. Targeted help does show the boards the student gets wrong.
- **Loss weighting still leaves c at 2.6 and predicts worse.**
  - With c and γ free, loss material needs $c$ 2.6 and $\gamma$ 1.15, no closer to 1 than the counts.
  - Plain loss material (exactly the weighted loss the network minimizes) does no better than plain counts: r 0.87 against 0.87 in sample, and 0.81 against 0.82 with each teacher held out.
  - With c and γ free, loss predicts held-out teachers worse than counts: r 0.93 against 0.96, rmse 0.13 against 0.10. It also predicts bayes3 from no teacher and the random teachers worse: r 0.80 against 0.92. The policy term alone is worse still.
- **Loss pushes the dilemma the wrong way.** Heavy targeted help's shown examples carry high loss, so loss material credits heavy help *more*. Its predicted θ 0.9 cost shrinks to −0.07 at agency 0, against −0.13 from counts and −0.24 observed (figure, right).
- **Neither do the other two explanations for c.**
  - **Steps:** Adam steps taken, shared by loss, fit about as well as counts (held-out r 0.955), but still need $c$ 2.2.
  - **Wandering:** with own examples split into a path's worth and the wandering beyond it, the fit counts the wandering at δ 0.87. It's worth nearly as much as on-path examples, and $c$ stays at 2.3.

  So per unit of examples, loss or steps, a shown move teaches the tests about 2.5 times what one of the student's own moves does at agency 1. The one candidate left is what the label *is*: a shown move is the shortest-path move from a board the student actually reached, which is what the tests score. A hindsight label on the student's own path is a shortest route through boards it happened to visit.
- **Where the heavy-help cost must come from.** At agency 0, shown examples carry no weight, so θ 0.9's cost there can only come from the student's own material.
  - Own examples drop by 19% (19,355 to 15,691 per interval), own loss by 18%, and steps taken by 18%. All three measures fall about alike.
  - The model turns that drop into a cost of 0.13; the observed cost is 0.24 here and 0.32 in p6. So the student's learning responds to losing its own play more steeply than to the material lost.
  - The own play that heavy help removes is on the puzzles the student would have struggled with. A per-bin test score may be too coarse a state to see what those puzzles teach.

**Next.**
- Option 1: go on to step 3 with the count model, which is still the best. Expect it to under-price heavy targeted help at low agency by about half.
- Option 2: first find what removing own play costs. At agency 0 the cost is all own play, which makes it the clean case. Two ways to test it: weight own examples by how hard the puzzle was for the student (bayes3's solve-alone estimate is saved per puzzle), or give the model a richer state than the per-bin test score.

## Diagnosis: why does losing own play cost more than the model says?

At agency 0, shown moves carry no weight, so bayes3 θ 0.9's cost against θ 0.45 there can only come from the student's own play. The model above turns the drop in own material into a cost of about 0.10–0.13; the observed cost is 0.24 to 0.32. Where does the model go wrong?

1. **Where the misfit sits:** the model's errors per condition and length bin.
2. **Is the test score a noisy state?** The model is fit one interval at a time, starting each from the *observed* test scores at its start. Those are measured on 50 puzzles × 3 attempts per bin, so they're noisy. A noisy starting state makes the fit mistake noise for dynamics: a student who happened to test badly gains more in the next interval, which looks like a steeper pull toward the floor. That steeper pull leaves less for material to explain. If so, successive interval gains should be negatively correlated.
3. **Fit along whole trajectories:** the same model, but each session is chained from its first test through all four intervals, and fit to every later test. The state is then the model's own estimate, not a noisy measurement. That's also how a planner will use it, rolling forward from the start.
4. **A clean test:** fit on the p6 runs, then predict the 20 fresh loss-logged runs.

In [ ]:
def per_bin(p, d, kind="full"):
    "Per session and length bin: learning predicted (chained from the first test) and observed."
    rows = []
    for key, g in d.groupby(["agency", "teacher", "student"]):
        g = g.sort_values("start")
        s, m = g[cols("s0")].values[:1], material(p, g, kind)
        for i in range(len(g)):
            s = step(p, s, m[i:i + 1])
        first, last = g[cols("s0")].values[0], g[cols("s1")].values[-1]
        rows += [{"agency": key[0], "teacher": key[1], "bin": f"{lo}-{hi}", "own": g[f"own_{b}"].sum(),
                  "error": (first[b] - s[0][b]) - (first[b] - last[b])} for b, (lo, hi) in enumerate(BINS)]
    return pd.DataFrame(rows)


where = per_bin(fits["full"], data)
pick = where[(where.agency == 0.0) | (where.teacher == "random:1.0") | (where.agency == 0.1) & (where.teacher == "random:0.3")]
ORDER = [f"{lo}-{hi}" for lo, hi in BINS]
display(pick.groupby(["teacher", "agency", "bin"])["error"].mean().unstack("bin")[ORDER].round(2)
        .rename_axis(columns="predicted − observed learning, by test length bin"))
display(pick[pick.agency == 0.0].groupby(["teacher", "bin"])["own"].mean().unstack("bin")[ORDER].round(0)
        .rename_axis(columns="own examples per session, agency 0"))

In [ ]:
def trajectories(d):
    "Sessions in order, 4 intervals each: (intervals sorted, first scores (sessions, bins), later scores (sessions, 4, bins))."
    d = d.sort_values(["agency", "teacher", "student", "start"]).reset_index(drop=True)
    assert (d.groupby(["agency", "teacher", "student"]).size() == 4).all()
    n = len(d) // 4
    return d, d[cols("s0")].values.reshape(n, 4, NB)[:, 0], d[cols("s1")].values.reshape(n, 4, NB)


def successive_gains(d):
    "Correlation of a session's successive interval gains, per bin: test noise in the shared score makes it negative."
    _, first, later = trajectories(d)
    scores = np.concatenate([first[:, None], later], 1)
    gains = scores[:, :-1] - scores[:, 1:]
    out = []
    for b, (lo, hi) in enumerate(BINS):
        g1, g2 = gains[:, :-1, b].ravel(), gains[:, 1:, b].ravel()
        cov = np.cov(g1, g2)[0, 1]
        out.append({"bin": f"{lo}-{hi}", "correlation of successive gains": np.corrcoef(g1, g2)[0, 1],
                    "noise sd implied (√−cov)": np.sqrt(max(-cov, 0)), "sd of interval gains": gains[:, :, b].std()})
    return pd.DataFrame(out).set_index("bin")


display(successive_gains(data).round(3))

In [ ]:
def fit_trajectory(d, kind="full"):
    "Least-squares parameters, chaining each session from its first test through its 4 intervals."
    d, first, later = trajectories(d)
    x0, lo, hi = start(kind)

    def residuals(p):
        m = material(p, d, kind).reshape(len(first), 4, NB)
        s, out = first, []
        for i in range(4):
            s = step(p, s, m[:, i])
            out.append(s)
        return (np.stack(out, 1) - later).ravel()
    return least_squares(residuals, x0, bounds=(lo, hi)).x


fits["trajectory"] = fit_trajectory(data)
floor, rate, q, rho, c, gamma, _ = unpack(fits["trajectory"])
display(pd.DataFrame([np.r_[unpack(fits[k])[0], unpack(fits[k])[1], unpack(fits[k])[2:6]] for k in ("full", "trajectory")],
                     index=["fit per interval", "fit along trajectories"], columns=P_NAMES[:-1]).round(3))

by_fit = {}
for name, fitter in (("per interval", fit), ("trajectory", fit_trajectory)):
    by_fit[name] = {
        "in sample": sessions(fitter(data), data),
        "leave one teacher out": pd.concat([held_out(lambda d, t=t: d.teacher != t, lambda d, t=t: d.teacher == t,
                                                     fitter=fitter) for t in SPECS]),
        "leave one agency out": pd.concat([held_out(lambda d, a=a: d.agency != a, lambda d, a=a: d.agency == a,
                                                    fitter=fitter) for a in AGENCIES]),
        "none + random → bayes3": held_out(lambda d: d.family != "bayes3", lambda d: d.family == "bayes3",
                                           fitter=fitter),
        "p6 → fresh loss-logged runs": sessions(fitter(data), logged),
    }
display(pd.DataFrame([{"fit": name, **scorecard(test, s)} for name, tests in by_fit.items() for test, s in tests.items()])
        .set_index(["test", "fit"]).sort_index(level=0, sort_remaining=False).round(3))

traj_sig = signatures(by_fit["trajectory"]["leave one teacher out"])
old_sig = signatures(by_fit["per interval"]["leave one teacher out"])
display(pd.DataFrame({(n, col): (old_sig if col == "per interval" else traj_sig)[(n, "predicted" if col != "observed" else col)]
                      for n in ("θ 0.9 − θ 0.45", "θ 0.97 − θ 0.45", "plateau (θ 0.25-0.75)", "random 100% − 30%")
                      for col in ("observed", "per interval", "trajectory")}).round(2))
fresh_sig = {name: logged_signatures(by_fit[name]["p6 → fresh loss-logged runs"]) for name in by_fit}
display(pd.DataFrame({(n, col): (fresh_sig["per interval"][(n, "observed")] if col == "observed"
                                 else fresh_sig[col][(n, "predicted")])
                      for n in ("θ 0.9 − θ 0.45", "θ 0.97 − θ 0.45", "random 100% − 30%")
                      for col in ("observed", "per interval", "trajectory")}).round(2)
        .rename_axis(index="agency (fresh runs, fit on p6)"))

In [ ]:
def diagnosis_figure(path):
    "Left: θ 0.9's cost by agency, observed and predicted by each fit. Right: the fresh runs predicted from p6."
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
    name = "θ 0.9 − θ 0.45"
    v = old_sig[(name, "observed")]
    a1.plot(v.index, v.values, "-o", color="#08306b", lw=2, ms=6, mec="white", mew=1, label="observed, p6 runs")
    v = fresh_sig["per interval"][(name, "observed")]
    a1.plot(v.index, v.values, "o", color="#08306b", mfc="white", mew=1.6, ms=7, label="observed, fresh runs")
    for label, sig, ls in (("fit per interval", old_sig, "--"), ("fit along trajectories", traj_sig, "-")):
        v = sig[(name, "predicted")]
        a1.plot(v.index, v.values, ls, color="#2a78d6", lw=2, label=f"predicted, {label}")
    a1.axhline(0, color="#5f5e5a", lw=1)
    a1.set_xticks(AGENCIES, [f"{a:g}" for a in AGENCIES])
    a1.set_xlabel("Agency")
    a1.set_ylabel("Learning, θ 0.9 minus θ 0.45")
    a1.legend(frameon=False, fontsize=8.5, loc="lower right")
    a1.set_title("The cost of heavy targeted help (each teacher held out)", fontsize=10, loc="left")
    lim = (0.4, 2.35)
    a2.plot(lim, lim, color="#b5b4ad", lw=1, zorder=0)
    for label, mk, fill in (("per interval", "o", "white"), ("trajectory", "o", None)):
        m = conditions(by_fit[label]["p6 → fresh loss-logged runs"])
        for (a, t), row in m.iterrows():
            a2.plot(row.observed, row.predicted, mk, color=AGENCY_COLORS[a], mfc=fill or AGENCY_COLORS[a],
                    mec=AGENCY_COLORS[a] if fill else "white", mew=1.4 if fill else 1, ms=7)
        a2.plot([], [], mk, color="#5f5e5a", mfc=fill or "#5f5e5a", mec="#5f5e5a" if fill else "white",
                label=f"fit {label}" if label == "per interval" else "fit along trajectories")
    a2.set_xlim(lim), a2.set_ylim(lim)
    a2.set_xlabel("Observed learning, fresh runs")
    a2.set_ylabel("Predicted from the p6 runs")
    a2.legend(frameon=False, fontsize=8.5, loc="upper left")
    a2.set_title("20 fresh conditions (shade: agency)", fontsize=10, loc="left")
    for ax in (a1, a2):
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#b5b4ad")
        ax.tick_params(colors="#5f5e5a")
        ax.grid(axis="y", color="#e6e5e0", lw=0.8)
        ax.set_axisbelow(True)
    fig.suptitle("The same learning model, fit per interval or along whole trajectories: 8 students per condition",
                 fontsize=10, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


diagnosis_figure(os.path.join(FIGURES, "trajectory_fit.png"))

**Result: most of the under-priced cost came from how the model was fit, not from what it leaves out. Fit along whole trajectories, the same model predicts held-out teachers and fresh runs clearly better, and gets more of heavy help's cost at low agency. Use the trajectory fit from here on.**

- **Where the misfit sits.** Fit per interval, the model over-predicts the conditions that learn mostly from their own play at low agency, and mostly on the longer tests:
  - no teacher at agency 0, by 0.26 to 0.38 on the three longer bins;
  - random 30% at agency 0.1, by 0.34 and 0.48 on the two longest;
  - θ 0.9 at agency 0, by 0.18 and 0.24 on the two longest.

  Observed learning on 24–26-move tests rises almost linearly with own examples at agency 0: 14.5k, 20.9k and 29k per session give 1.69, 2.21 and 3.07. The per-interval fit levels off much sooner.
- **The test score is a noisy state.** Successive interval gains are negatively correlated in three of the four bins (−0.14, −0.09, −0.07; +0.14 in 14–16). That implies test noise of about 0.08–0.13 in a bin's score, against gains with an sd of 0.26–0.6. This is suggestive, not decisive. The decisive evidence is the out-of-sample test below.
- **Fit along trajectories, the same model with the same 12 parameters does better on every test:**

  | | fit per interval | fit along trajectories |
  |---|---|---|
  | each teacher held out | r 0.941, rmse 0.100 | r 0.962, rmse 0.086 |
  | each agency held out | r 0.941, rmse 0.102 | r 0.970, rmse 0.075 |
  | fit on p6, predict the 20 fresh runs | r 0.957, rmse 0.099 | r 0.976, rmse 0.073 |
  | none + random → bayes3 | r 0.906, bias −0.11 | r 0.909, bias −0.09 |

  - The parameters change in the direction noise would explain. Rates are 2 to 3 times lower and the long-bin floors lower (4.06 against 4.44), so the score keeps responding to material instead of saturating early.
  - c stays at 2.35. With γ 0.77, a shown example now equals an own one at agency about 0.33.
- **The dilemma's signatures move toward observed:**
  - θ 0.9 − θ 0.45 at agency 0: −0.18, against −0.10 before; observed −0.32 in p6 and −0.24 in the fresh runs. At agency 0.1 it now matches (−0.15 against −0.15).
  - θ 0.97 − θ 0.45 at agency 0.5: −0.21, against −0.15 before; observed −0.30 and −0.36.
  - The plateau's rise with agency: 2.00 → 2.08, against 1.99 → 2.05; observed 2.03 → 2.15.
- **What's still missing.**
  - **Heavy help at agency 0.3** is still under-priced: −0.11 predicted against −0.19 and −0.23 observed.
  - **Random 30% at low agency** is still over-predicted, so the random gap stays too large: −1.29 against −0.87 at agency 0.1. Own play mixed with random shown moves teaches less than its counts say.
- **Tried outside the notebook, and dropped** (scripts in the session scratchpad):
  - per-bin c;
  - a separate floor for shown material (the idea that shown moves teach only the on-path moves);
  - curve shapes on cumulative material (S-shaped exponential, power law).

  Fit per interval, none of these moved θ 0.9's predicted cost at agency 0 (all −0.10). Fit along trajectories, the separate floor improved the random gap (−1.12 at agency 0.1) and held-out teachers (r 0.969). But on the fresh runs it barely beat the plain trajectory fit (r 0.978 against 0.976), so it isn't worth four more parameters.

**For the POMDP.** The planner will roll the model forward from the start of the session, so it should be fit the same way. Use `fit_trajectory`. Its remaining error is that it under-prices heavy targeted help at low agency by about a third. So expect a POMDP teacher planned on it to show somewhat more than is best when agency is low.

## Does a θ that changes beat a constant one?

bayes3 already adapts: its threshold is on the student's predicted chance of solving alone, so it helps less as the student improves. A POMDP teacher could add more: help that depends on how much of the session is left, or on how long the puzzle is. Before building one, a cheap test asks whether either kind of variation pays on the real student.

Two new teachers in `teach.py` switch θ between 0.25 and 0.9, in both directions:

- **By time (`TimedBayes3`):** one θ for the first half of the session (1080 puzzles), the other for the second half.
- **By length (`LengthBayes3`):** one θ on puzzles shorter than 18 moves (the 9–11 and 14–16 bins), the other on the 19–21 and 24–26 bins.

Each schedule runs at agency 0.1 and 0.5 on the same 8 students as constant θ 0.45, with the loss-logged runs' settings and the hash seed fixed (runs `schedule_agency0.1` and `schedule_agency0.5`). It is compared with θ 0.45 student by student.

- **Decision rule:** if no schedule beats θ 0.45 by more than the noise (about 0.05), a POMDP teacher that varies help along these lines is unlikely to beat bayes3 either.
- **Noise:** θ 0.45 was also run in the loss-logged runs. Same students, other runs, so the difference between the two is noise alone.
- **The learning model:** does the trajectory fit, trained on the p6 runs only, predict which schedules cost most? A planner will lean on exactly that.

In [ ]:
SCHED_AGENCIES = [0.1, 0.5]
SCHED_SPECS = {"bayes3:0.45": "θ 0.45 throughout",
               "bayes3-time:0.9,late=0.25": "θ 0.9, then 0.25 (more help early)",
               "bayes3-time:0.25,late=0.9": "θ 0.25, then 0.9 (more help late)",
               "bayes3-length:0.9,long=0.25": "θ 0.9 short, 0.25 long (more help on short)",
               "bayes3-length:0.25,long=0.9": "θ 0.25 short, 0.9 long (more help on long)"}
HALF = 1080  # puzzles in the first half of the session


def run_intervals(runs):
    "Intervals as in intervals(), for [(agency, run, specs)]; plus the shown share in each half of the session."
    rows, halves = [], []
    for a, run, specs in runs:
        for spec in specs:
            e = sa.entry(run, spec)
            if e is None:
                continue
            for s, record in e["sessions"].items():
                tests = e["results"][s]
                points = sorted(tests)
                scores = {n: bin_scores(tests[n], e["config"]) for n in points}
                for n0, n1 in zip(points, points[1:]):
                    row = {"agency": a, "teacher": spec, "student": s, "start": n0}
                    chunk = record[n0:n1]
                    for b in range(NB):
                        row[f"s0_{b}"], row[f"s1_{b}"] = scores[n0][b], scores[n1][b]
                        inb = [x for x in chunk if bin_of(x["distance"]) == b]
                        row[f"own_{b}"] = sum(x["examples"] * (1 - x["taught"] / x["moves"]) for x in inb if x["solved"])
                        row[f"mixed_{b}"] = sum(x["examples"] * (1 - x["taught"] / x["moves"]) for x in inb
                                                if x["solved"] and x["taught"])
                        row[f"shown_{b}"] = sum(x["examples"] * (x["taught"] / x["moves"] if x["solved"] else 1)
                                                for x in inb)
                    rows.append(row)
                share = lambda r: sum(x["taught"] for x in r) / sum(x["moves"] for x in r)
                halves.append({"agency": a, "teacher": spec, "student": s, "first half": share(record[:HALF]),
                               "second half": share(record[HALF:]), "session": share(record)})
    df = pd.DataFrame(rows)
    df["family"] = df.teacher.str.split(":").str[0]
    return df, pd.DataFrame(halves)


sched, sched_halves = run_intervals([(a, f"schedule_agency{a}", SCHED_SPECS) for a in SCHED_AGENCIES])
print(len(sched), "intervals:", sched.groupby(["agency", "teacher"]).ngroups, "conditions x 8 students x 4")
sched_material = sched.assign(own=sched[cols("own")].sum(axis=1), shown=sched[cols("shown")].sum(axis=1),
                              own_long=sched[["own_2", "own_3"]].sum(axis=1))
sched_material = (sched_material.groupby(["agency", "teacher", "student"])[["own", "shown", "own_long"]].sum()
                  .groupby(["agency", "teacher"]).mean() / 1000)
display(sched_halves.groupby(["agency", "teacher"])[["first half", "second half", "session"]].mean()
        .join(sched_material).rename(index=SCHED_SPECS).round(3)
        .rename(columns={"own": "own examples (k)", "shown": "shown examples (k)",
                         "own_long": "own examples, 19-26 moves (k)"})
        .rename_axis(columns="share of moves shown; examples per session"))

In [ ]:
def paired(per_session, base_session=None):
    "Each schedule minus θ 0.45, student by student: observed (mean, se, students who did better) and predicted."
    s = per_session.astype(float).reset_index()
    out = []
    for a in SCHED_AGENCIES:
        base = (base_session if base_session is not None else s)
        base = base[(base.agency == a) & (base.teacher == "bayes3:0.45")].set_index("student")
        for t in [t for t in SCHED_SPECS if t in set(s.teacher)]:
            if t == "bayes3:0.45" and base_session is None:
                continue
            x = s[(s.agency == a) & (s.teacher == t)].set_index("student")
            diff = x[["observed", "predicted"]] - base[["observed", "predicted"]]
            out.append({"agency": a, "teacher": t, "observed": diff.observed.mean(),
                        "se": diff.observed.std() / np.sqrt(len(diff)), "students better": int((diff.observed > 0).sum()),
                        "predicted": diff.predicted.mean()})
    return pd.DataFrame(out)


sched_sessions = sessions(fits["trajectory"], sched)
sched_paired = paired(sched_sessions)
# noise: θ 0.45 in the loss-logged runs, against θ 0.45 here
replicate = paired(sessions(fits["trajectory"], logged[logged.agency.isin(SCHED_AGENCIES)
                                                       & (logged.teacher == "bayes3:0.45")]),
                   base_session=sched_sessions.astype(float).reset_index())
display(pd.concat([sched_paired, replicate.assign(teacher="θ 0.45, loss-logged runs (noise)")])
        .replace({"teacher": SCHED_SPECS}).set_index(["agency", "teacher"]).round(3)
        .rename(columns={"observed": "observed Δ", "predicted": "predicted Δ (fit on p6)"})
        .rename_axis(columns="learning minus θ 0.45's, same students"))
print(scorecard("schedule runs, fit on p6", sched_sessions))
for a in SCHED_AGENCIES:
    p = sched_paired[sched_paired.agency == a]
    print(f"agency {a}: rank correlation of predicted and observed Δ {p.observed.rank().corr(p.predicted.rank()):.2f}")

In [ ]:
def schedule_figure(path):
    "Each schedule minus θ 0.45, per agency: observed (± se) and predicted by the trajectory fit."
    order = list(SCHED_SPECS)[1:]
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.1), sharey=True)
    for ax, a in zip(axes, SCHED_AGENCIES):
        p = sched_paired[sched_paired.agency == a].set_index("teacher").loc[order]
        y = np.arange(len(order))[::-1]
        ax.axvspan(-0.05, 0.05, color="#efeee9", zorder=0)
        ax.axvline(0, color="#5f5e5a", lw=1)
        ax.errorbar(p.observed, y, xerr=p.se, fmt="o", color="#08306b", ms=7, mec="white", mew=1, elinewidth=1.6,
                    capsize=0, label="observed ± se (8 students)")
        ax.plot(p.predicted, y, "D", color="#2a78d6", mfc="white", mew=1.6, ms=7, label="predicted, fit on p6")
        ax.set_yticks(y, [SCHED_SPECS[t] for t in order])
        ax.set_xlim(-0.32, 0.15)
        ax.set_xlabel("Learning minus θ 0.45's, same students")
        ax.set_title(f"Agency {a:g}", fontsize=10, loc="left")
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color("#b5b4ad")
        ax.tick_params(colors="#5f5e5a", length=0)
        ax.grid(axis="x", color="#e6e5e0", lw=0.8)
        ax.set_axisbelow(True)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, fontsize=8.5, loc="upper right", ncol=2)
    fig.suptitle("No schedule beats constant θ 0.45 (shaded: ±0.05, the noise between two runs of θ 0.45)",
                 fontsize=10, x=0.01, ha="left")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


schedule_figure(os.path.join(FIGURES, "schedules.png"))

**Result: no schedule beats constant θ 0.45. Each one either ties or costs. The learning model, fit on p6 alone, predicts that each one costs, and ranks them well at agency 0.1. At agency 0.5 it puts the costs 2–3× too low.**

- **On the real student**, each schedule minus θ 0.45, same students (8):

  | schedule | agency 0.1 | agency 0.5 |
  |---|---|---|
  | more help late (θ 0.25, then 0.9) | +0.01 ± 0.08 | −0.06 ± 0.08 |
  | more help on short puzzles | −0.06 ± 0.05 | −0.08 ± 0.05 |
  | more help early (θ 0.9, then 0.25) | −0.08 ± 0.04 | −0.16 ± 0.09 |
  | more help on long puzzles | −0.15 ± 0.04 | −0.13 ± 0.07 |
  | *θ 0.45 again (loss-logged runs): noise* | −0.04 ± 0.06 | +0.03 ± 0.06 |

  - Only more help late is within the noise, and it doesn't come out ahead at either agency. By the decision rule, varying θ by time or puzzle length doesn't pay.
  - The order is mostly how much each schedule shows in total: 3.6% of moves for θ 0.45, 4–9% for the schedules. The one exception is more help late. It shows more than more help on short puzzles (5.2% against 4.3%), yet costs no more.
  - Extra help on long puzzles costs the most. It cuts own examples on the 19–26-move puzzles from about 53k to 40k per session, and the loss falls mostly on those bins (−0.36 there at agency 0.1).
  - Extra help early costs more than the same switch made late, by about 0.09 at agency 0.1 and 0.10 at 0.5. Some of that is dose: early, θ 0.9 shows 12–13% of moves, against 8–9% late, when the student needs less. This direction is the opposite of fading scaffolding, where help is front-loaded.
- **The learning model's prediction** (the trajectory fit, trained on the p6 runs only):
  - **Direction:** every schedule is predicted to lose, so a planner on this model wouldn't choose them.
  - **Order:** rank correlation 0.8 at agency 0.1, 0.6 at 0.5. Per session, r 0.85 overall and 0.76 within an agency.
  - **Size:** right at agency 0.1, for example −0.13 predicted against −0.15 for more help on long puzzles. Too small at 0.5: −0.04 against −0.16 for more help early, and −0.06 against −0.13 for more help on long puzzles. That's the same under-pricing of help at middling agency as before.
  - In the model, each cost comes from material: more shown examples and fewer own ones. It has no separate term for when help comes.

**For the POMDP.** Along the two dimensions a planner would most easily exploit, time left and puzzle length, constant θ 0.45 is already about as good as tested variations. A POMDP solution is still useful as a model-based benchmark for that claim, and it can test the one thing these schedules don't: playing in order to learn about the student. Planned on this model, though, it will help too readily at middling agency.

## An upper bound: the best teacher if the student's state were known

A POMDP teacher can't do better than a teacher that sees the student's knowledge state exactly. In the learning model, that teacher faces an ordinary MDP, and its best value is an upper bound on any POMDP solution in the same model. Here the bound is simpler still: every student starts from the same network, so the first test gives the same scores, and the model is deterministic. The state at every point is then fixed by the plan, so the best teacher is the best fixed plan.

To plan, the learning model needs a **play model**: for a given state and teacher, how many own and shown examples an interval produces.

- **Actions:** the eight teachers already run (none, bayes3 θ 0.25 to 0.97, random 30% and 100%). Each interval can use a different one, and each length bin too.
- **Play model:** per action and bin, own and shown examples are a straight line in the bin's model state. The first interval is separate: it's the same state for everyone, so it's just the mean. Play hardly depends on agency (it changes learning, not play), so agencies are pooled. Fit on the p6 runs.
- **Checks:** the simulator (play model, then learning model, from the first test on) has to reproduce the constant teachers in p6, and predict the schedule runs out of sample, where θ differs by bin and by half.
- **The bound:** search all plans that change the teacher per interval (8⁴), and per interval × short/long (64⁴, pruned). Then improve per interval × bin by coordinate ascent. Compare with constant θ 0.45.
- **Robustness:** resample the 8 students, refit both models, and recompute.

In [ ]:
ACTIONS = ["none", "bayes3:0.25", "bayes3:0.45", "bayes3:0.75", "bayes3:0.9", "bayes3:0.97", "random:0.3", "random:1.0"]


def with_states(p, d):
    "Intervals in session order, with the model state at each interval's start (chained from the first test) and its index."
    d, first, _ = trajectories(d)
    m = material(p, d, "full").reshape(len(first), 4, NB)
    s, states = first, []
    for i in range(4):
        states.append(s)
        s = step(p, s, m[:, i])
    st = np.stack(states, 1).reshape(-1, NB)
    d = d.copy()
    for b in range(NB):
        d[f"m0_{b}"] = st[:, b]
    d["interval"] = d.groupby(["agency", "teacher", "student"]).cumcount()
    return d


def fit_play(d):
    "Per action, bin and kind (own, shown): the first interval's mean, and a line in the bin's model state after it."
    P = {}
    for a in ACTIONS:
        g = d[d.teacher == a]
        g0, g1 = g[g.interval == 0], g[g.interval > 0]
        for b in range(NB):
            for k in ("own", "shown"):
                X = np.c_[np.ones(len(g1)), g1[f"m0_{b}"]]
                P[a, b, k] = (g0[f"{k}_{b}"].mean(), np.linalg.lstsq(X, g1[f"{k}_{b}"].values, rcond=None)[0])
    return P


def play(P, actions, s, interval):
    "Own and shown examples per bin, (n, NB), for per-bin actions at states s (n, NB)."
    own, shown = np.zeros_like(s), np.zeros_like(s)
    for b in range(NB):
        for k, out in (("own", own), ("shown", shown)):
            mean0, (c0, c1) = P[actions[b], b, k]
            out[:, b] = mean0 if interval == 0 else np.maximum(c0 + c1 * s[:, b], 0)
    return own, shown


def simulate(P, p, plan, alpha, s0):
    "Learning (mean over bins) of a plan: 4 intervals of per-bin actions, from the first-test scores s0."
    *_, c, gamma, _ = unpack(p)
    s = np.atleast_2d(s0).astype(float)
    for i, actions in enumerate(plan):
        own, shown = play(P, actions, s, i)
        s = step(p, s, own + c * alpha ** gamma * shown)
    return float((np.atleast_2d(s0) - s).mean())


constant = lambda a: [[a] * NB] * 4
play_data = with_states(fits["trajectory"], data)
PLAY = fit_play(play_data)
START = play_data[play_data.interval == 0][cols("s0")].values.mean(0)
print("first-test scores, every student:", START.round(2), " sd across sessions:",
      play_data[play_data.interval == 0][cols("s0")].values.std(0).round(3))

fit_quality = {}
for k in ("own", "shown"):
    pred = np.concatenate([play(PLAY, [r.teacher] * NB, np.array([[getattr(r, f"m0_{b}") for b in range(NB)]], float), r.interval)
                           [0 if k == "own" else 1] for r in play_data.itertuples()])
    obs = play_data[cols(k)].values
    fit_quality[k] = 1 - ((obs - pred) ** 2).sum(0) / ((obs - obs.mean(0)) ** 2).sum(0)
display(pd.DataFrame(fit_quality, index=[f"{lo}-{hi}" for lo, hi in BINS]).T.round(3)
        .rename_axis(columns="play model R², per interval and session"))

observed = conditions(sessions(fits["trajectory"], data))["observed"]
sim = pd.DataFrame([{"agency": a, "teacher": t, "observed": v, "simulated": simulate(PLAY, fits["trajectory"], constant(t), a, START)}
                    for (a, t), v in observed.items()])
print(f"constant teachers, p6 (in sample): r {np.corrcoef(sim.observed, sim.simulated)[0, 1]:.3f}, "
      f"rmse {np.sqrt(((sim.observed - sim.simulated) ** 2).mean()):.3f}")
display(sim.set_index(["teacher", "agency"]).stack().unstack(["agency"]).round(2)
        .rename_axis(columns="learning by agency: observed, and simulated from the first test"))

In [ ]:
SCHED_PLANS = {"bayes3:0.45": constant("bayes3:0.45"),
               "bayes3-time:0.9,late=0.25": [["bayes3:0.9"] * NB] * 2 + [["bayes3:0.25"] * NB] * 2,
               "bayes3-time:0.25,late=0.9": [["bayes3:0.25"] * NB] * 2 + [["bayes3:0.9"] * NB] * 2,
               "bayes3-length:0.9,long=0.25": [["bayes3:0.9"] * 2 + ["bayes3:0.25"] * 2] * 4,
               "bayes3-length:0.25,long=0.9": [["bayes3:0.25"] * 2 + ["bayes3:0.9"] * 2] * 4}
sim_paired = sched_paired.copy()
for a in SCHED_AGENCIES:
    base = simulate(PLAY, fits["trajectory"], SCHED_PLANS["bayes3:0.45"], a, START)
    for i in sim_paired.index[sim_paired.agency == a]:
        t = sim_paired.at[i, "teacher"]
        sim_paired.at[i, "simulated"] = simulate(PLAY, fits["trajectory"], SCHED_PLANS[t], a, START) - base
display(sim_paired.replace({"teacher": SCHED_SPECS}).set_index(["agency", "teacher"])
        [["observed", "se", "predicted", "simulated"]].round(3)
        .rename(columns={"observed": "observed Δ", "predicted": "learning model, observed material",
                         "simulated": "simulator (play model too)"})
        .rename_axis(columns="schedule minus θ 0.45 (out of sample)"))

In [ ]:
def search(p, P, alpha, options, prune=20000):
    "Best plan over 4 intervals, each choosing one of options (per-bin action tuples); exhaustive unless pruned."
    *_, c, gamma, _ = unpack(p)
    s, plans = START[None].copy(), np.zeros((1, 0), int)
    for i in range(4):
        n, k = len(s), len(options)
        S, A = np.repeat(s, k, 0), np.array(options * n)
        own, shown = np.zeros_like(S), np.zeros_like(S)
        for b in range(NB):
            for a in set(A[:, b]):
                idx = A[:, b] == a
                o, sh = play(P, [a] * NB, S[idx], i)
                own[idx, b], shown[idx, b] = o[:, b], sh[:, b]
        s = step(p, S, own + c * alpha ** gamma * shown)
        plans = np.c_[np.repeat(plans, k, 0), np.tile(np.arange(k), n)]
        if i < 3 and len(s) > prune:  # keep the plans furthest along so far
            keep = np.argsort(s.mean(1))[:prune]
            s, plans = s[keep], plans[keep]
    learn = (START - s).mean(1)
    j = learn.argmax()
    return learn[j], [list(options[h]) for h in plans[j]]


def ascend(p, P, alpha, plan):
    "Coordinate ascent over each interval and bin's action, from plan."
    v, improved = simulate(P, p, plan, alpha, START), True
    while improved:
        improved = False
        for i in range(4):
            for b in range(NB):
                for a in ACTIONS:
                    trial = [list(x) for x in plan]
                    trial[i][b] = a
                    vt = simulate(P, p, trial, alpha, START)
                    if vt > v + 1e-9:
                        v, plan, improved = vt, trial, True
    return v, plan


short_name = lambda a: a.replace("bayes3:", "θ ").replace("random:", "random ")
UNIFORM = [tuple([a] * NB) for a in ACTIONS]
GROUPED = [tuple([a1] * 2 + [a2] * 2) for a1 in ACTIONS for a2 in ACTIONS]
bound_rows, bound_plans = [], {}
for a in AGENCIES:
    const = {t: simulate(PLAY, fits["trajectory"], constant(t), a, START) for t in ACTIONS}
    best = max(const, key=const.get)
    v_interval, plan_interval = search(fits["trajectory"], PLAY, a, UNIFORM)
    v_grouped, plan_grouped = search(fits["trajectory"], PLAY, a, GROUPED)
    v_bin, plan_bin = ascend(fits["trajectory"], PLAY, a, plan_grouped)
    bound_plans[a] = plan_bin
    bound_rows.append({"agency": a, "θ 0.45": const["bayes3:0.45"], "best constant": short_name(best),
                       "its learning": const[best], "per interval": v_interval, "per interval × short/long": v_grouped,
                       "per interval × bin (bound)": v_bin, "headroom over θ 0.45": v_bin - const["bayes3:0.45"]})
bound_table = pd.DataFrame(bound_rows).set_index("agency")
display(bound_table.round(3).rename_axis(columns="simulated learning: θ 0.45, the best constant teacher, and the best plans"))
display(pd.DataFrame({a: [" ".join(short_name(x).replace("θ ", "") for x in interval) for interval in plan]
                      for a, plan in bound_plans.items()}, index=[f"interval {i + 1}" for i in range(4)]).T
        .rename_axis(index="agency", columns="best plan: action per bin (9-11, 14-16, 19-21, 24-26)"))

In [ ]:
N_BOOT = 50  # each resample refits the learning and play models (about a second)


def bootstrap_bound(n, agencies=(0.0, 0.1, 0.3, 0.5, 1.0), seed=0):
    "Resample students, refit the learning and play models, and recompute headroom (coordinate ascent from the best constant)."
    rng, students, rows = np.random.default_rng(seed), sorted(data.student.unique()), []
    for rep in range(n):
        parts = [data[data.student == s].assign(student=f"{s}#{j}") for j, s in enumerate(rng.choice(students, len(students)))]
        d = pd.concat(parts, ignore_index=True)
        p = fit_trajectory(d)
        P = fit_play(with_states(p, d))
        for a in agencies:
            const = {t: simulate(P, p, constant(t), a, START) for t in ACTIONS}
            best = max(const, key=const.get)
            v, _ = ascend(p, P, a, constant(best))
            rows.append({"rep": rep, "agency": a, "best constant": short_name(best), "headroom": v - const["bayes3:0.45"]})
    return pd.DataFrame(rows)


boot = bootstrap_bound(N_BOOT)
display(boot.groupby("agency").headroom.describe(percentiles=[0.5, 0.9])[["mean", "50%", "90%", "max"]].round(3)
        .rename_axis(columns=f"headroom over θ 0.45, {N_BOOT} resamples of students"))
display(boot.groupby("agency")["best constant"].value_counts().unstack(fill_value=0)
        .rename_axis(columns="best constant teacher, count of resamples"))

In [ ]:
def bound_figure(path):
    "Simulated learning by agency: each constant teacher, and the bound."
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    lines = [("bayes3:0.25", "#9ecae1", "-"), ("bayes3:0.75", "#4292c6", "-"), ("bayes3:0.9", "#2171b5", "-"),
             ("bayes3:0.97", "#08519c", "-"), ("random:0.3", "#8a8984", ":"), ("random:1.0", "#8a8984", "-."),
             ("none", "#8a8984", "--")]
    ax.plot(AGENCIES, bound_table["θ 0.45"], "-", color="#08306b", lw=2.6, label="θ 0.45")
    for t, color, ls in lines:
        v = [simulate(PLAY, fits["trajectory"], constant(t), a, START) for a in AGENCIES]
        ax.plot(AGENCIES, v, ls, color=color, lw=1.3, label=short_name(t))
    ax.plot(AGENCIES, bound_table["per interval × bin (bound)"], "o", color="#e6550d", ms=7, mec="white", mew=1,
            label="best plan (bound)", zorder=5)
    ax.set_ylim(1.5, 2.18)
    ax.set_xticks(AGENCIES, [f"{a:g}" for a in AGENCIES])
    ax.set_xlabel("Agency")
    ax.set_ylabel("Simulated learning over the session")
    ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1), title="constant teacher",
              title_fontsize=8.5, alignment="left")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#b5b4ad")
    ax.tick_params(colors="#5f5e5a")
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
    ax.set_title("In the model, no plan beats θ 0.45 at agency ≤ 0.5 (lines below 1.5 are cut off)", fontsize=10,
                 loc="left")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


bound_figure(os.path.join(FIGURES, "bound.png"))

**Result: in the model, constant θ 0.45 is the best teacher at agency 0 to 0.7, within 0.003. That holds even for a teacher that sees the student's state and can change its action every interval and in every length bin. A POMDP teacher planned on this model has essentially no room to beat bayes3 there. Only at agency 0.9 and 1 does the model find room, +0.02 and +0.04, and it finds it in random help. That's where the model disagrees with the real runs.**

- **The simulator holds up.**
  - The play model explains 96–97% of the variance in own examples and 98–99% in shown examples, per interval.
  - Rolled forward from the first test, play model and learning model together reproduce the 45 constant-teacher conditions at r 0.97 (rmse 0.07).
  - Out of sample, the schedule runs come out as before: every sign right, sizes right at agency 0.1 and too small at 0.5. Adding the play model barely changes the predictions.
- **The bound** (simulated learning; the per interval × bin plan is the bound):

  | agency | θ 0.45 | best constant | per interval | per interval × short/long | per interval × bin |
  |---|---|---|---|---|---|
  | 0 | 2.023 | θ 0.45 | 2.023 | 2.024 | 2.025 |
  | 0.1 | 2.034 | θ 0.45 | 2.034 | 2.034 | 2.036 |
  | 0.3 | 2.049 | θ 0.45 | 2.049 | 2.049 | 2.049 |
  | 0.5 | 2.060 | θ 0.45 | 2.060 | 2.060 | 2.060 |
  | 0.7 | 2.070 | θ 0.45 | 2.071 | 2.071 | 2.073 |
  | 0.9 | 2.080 | random 30% (2.080) | 2.093 | 2.098 | 2.099 |
  | 1 | 2.084 | random 30% (2.104) | 2.113 | 2.119 | 2.121 |

  - At agency ≤ 0.5 the best plans are θ 0.45 nearly everywhere. The only changes are θ 0.25 on one long bin in some intervals, worth 0.003 at most.
  - Across 50 resamples of the students, with both models refit each time, headroom at agency ≤ 0.5 never exceeds 0.009. At agency ≤ 0.1 the best constant teacher is θ 0.45 in 46 to 49 of the 50 resamples; at 0.3 and 0.5, in all 50.
- **Why θ 0.45 in the model: it gives the student the most own play to learn from.**
  - An unsolved puzzle teaches nothing. A little help turns failures into solves, and the student's own moves on them become examples. With no teacher, the student gets about 13k own examples per interval; with θ 0.25 or 0.45, about 19.6k.
  - More help displaces own moves: 18.6k at θ 0.75, 15.8k at θ 0.9.
  - A shown example is worth c·α^γ own ones: 0.4 at agency 0.1, 0.9 at 0.3, 1.4 at 0.5. That's too little to make up for the own play lost. So the optimum is the help level that keeps own play highest, and that doesn't depend on time or puzzle length.
  - At agency 0.9–1 a shown example is worth about 2.2–2.35 own ones, so the model starts to prefer showing. Random help shows the most, mostly on the short puzzles.
- **Where the bound is weak.**
  - **Only the teachers already run are actions.** A teacher that helps in some way none of these do, such as showing only at certain kinds of boards, isn't covered. Neither are θ values between the grid points, though θ 0.25–0.75 is flat on the real student anyway.
  - **High agency.** On the real student, θ 0.75 beats θ 0.45 at agency 0.9–1 (2.22 and 2.21 against 2.13 and 2.12), and random help doesn't. The model gets both wrong, so its bound there isn't credible.
  - **Agency 0.5.** The model puts help's cost too low there. That makes help look better than it is, which should pull the best plan toward more help. It still picks θ 0.45, so that conclusion doesn't rest on the error.
  - **Information.** In the model every student starts in the same state and moves deterministically, so watching the student adds nothing. Real students do vary from one session to the next, but within a condition the material hardly tracks a student's state (correlation about 0.1), so the room for adapting to the individual looks small.

**For the POMDP.** At the agencies where help has a cost, the model leaves no room above bayes3 θ 0.45, so a POMDP solution on it would reproduce bayes3. Solving one isn't worth it unless the student or the action set changes. The claim this supports: in this setting, the best help in the model is a fixed threshold on the student's chance of solving alone, set to keep their own play as high as possible.